# 第 5 章 · Tool Use（CrewAI）

同一件事换个框架：LangChain 用 `@tool` + `AgentExecutor`，CrewAI 用 `@tool` + `Agent/Task/Crew`。
**工具本身的写法几乎一样，变的是"谁来驱动它"**——CrewAI 把编排拆成三个显式对象：
Agent（谁做）、Task（做什么）、Crew（怎么组织）。

⚠️ **本机未安装 `crewai`**（体积较大），所以这一格没有执行输出。要跑需：

```bash
python -m venv ~/agentic_crewai_venv && ~/agentic_crewai_venv/bin/pip install crewai
```

建议装到独立 venv，**不要动主环境**。

kernel 需要能 import crewai 的那个环境。

In [ ]:
# pip install crewai
import os

from crewai import Agent, Task, Crew
from crewai.tools import tool          # ★ 本章新增（CrewAI 版）：和 LangChain 的 @tool 同一个角色

# 【适配】CrewAI 底层走 litellm，读的仍是 OPENAI_API_KEY / OPENAI_API_BASE。
#   本仓库 .env 已把这两个指向 DeepSeek，所以模型要写成 openai/ 前缀的形式。
#   书里注释掉的那两行就是原始写法：
#     os.environ["OPENAI_API_KEY"] = "..."
#     os.environ["OPENAI_MODEL_NAME"] = "gpt-4o"
os.environ.setdefault("OPENAI_MODEL_NAME", "openai/deepseek-flash")


@tool("Stock Price Lookup Tool")
def get_stock_price(ticker: str) -> float:
    """
    Fetches the latest simulated stock price for a given stock ticker symbol.
    Returns the price as a float. Raises a ValueError if the ticker is not found.
    """
    simulated_prices = {"AAPL": 178.15, "GOOGL": 1750.30, "MSFT": 425.50}
    price = simulated_prices.get(ticker.upper())
    if price is not None:
        return price
    # ★ 本章新增（CrewAI 版）：内部抛标准异常而不是返回字符串。
    #   上一节 LangChain 版讲的是"工具异常要变成数据"；这里是它的前提——
    #   工具自己别把错误吞成一句人话，否则 agent 分不清「拿到结果」和「出错了」。
    raise ValueError(f"Simulated price for ticker '{ticker.upper()}' not found.")


# Agent / Task / Crew 三层是 CrewAI 的骨架（第 7 章多 agent 会展开）。
financial_analyst_agent = Agent(
    role='Senior Financial Analyst',
    goal='Analyze stock data using provided tools and report key prices.',
    backstory="You are an experienced financial analyst adept at using data sources to find stock information. You provide clear, direct answers.",
    verbose=True,
    tools=[get_stock_price],
    allow_delegation=False,
)

analyze_aapl_task = Task(
    description=(
        "What is the current simulated stock price for Apple (ticker: AAPL)? "
        "Use the 'Stock Price Lookup Tool' to find it. "
        "If the ticker is not found, you must report that you were unable to retrieve the price."
    ),
    expected_output=(
        "A single, clear sentence stating the simulated stock price for AAPL. "
        "For example: 'The simulated stock price for AAPL is $178.15.' "
        "If the price cannot be found, state that clearly."
    ),
    agent=financial_analyst_agent,
)

financial_crew = Crew(
    agents=[financial_analyst_agent],
    tasks=[analyze_aapl_task],
    verbose=True,
)


def main():
    # 【适配】书里只检查 OPENAI_API_KEY；这里额外说明它必须指向 DeepSeek 且带 base_url。
    if not os.environ.get("OPENAI_API_KEY"):
        print("ERROR: OPENAI_API_KEY 未设置（本仓库应由 .env 提供）")
        return
    print("\n## Starting the Financial Crew...")
    result = financial_crew.kickoff()
    print("\n## Crew execution finished.\nFinal Result:\n", result)


if __name__ == "__main__":
    main()